In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, cross_validate
from sklearn.linear_model import LinearRegression
from scipy.stats import gaussian_kde

sys.path.insert(1, '../../ProcessEvents')
from config import CATCHMENT_LOOKUP_DICT, ENSEMBLE_MEMBERS, CATCHMENTS

from PrepDataFunctions import *



In [2]:
df = prep_for_modelling(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/all_catchments_with_topo.pkl", 
                        remove_events_with_zero_flooding = False)

print(len(df))
df = df[df['mean_sm_2_before_event'].notna()]
print(len(df))
# df[df['max_quartile_index_n'].isnull()][["max_precip", "max_quartile_index_n", "length", "total_acc", "length_of_active_rain"]]
# df = create_subset(df, 10000, 10)
subset_early = create_temporal_subset(df, "before", 2020)
subset_late = create_temporal_subset(df, "after", 2060)
# subset_late = create_subset(subset, 10000, 10)

104983
104970


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor 
from time import time

lr_model_pipeline = Pipeline(steps=[['scaler',StandardScaler()],['linear_regressor',LinearRegression()]])
rf_model_pipeline = Pipeline(steps=[['scaler',StandardScaler()],['rf_regressor', RandomForestRegressor(random_state = 1, n_jobs = 10)]])
xgb_model_pipeline = Pipeline(steps=[['scaler',StandardScaler()],['xgb_regressor',xgb.XGBRegressor(random_state=1, n_jobs = 16)]])
et_model_pipeline = Pipeline(steps=[['scaler',StandardScaler()],['et_regressor',ExtraTreesRegressor (random_state = 1, n_jobs = 16)]])

models_dict = {"linear_regressor": lr_model_pipeline, "xgb_regressor":xgb_model_pipeline, 
               "rf_regressor":rf_model_pipeline, 'et_regressor': et_model_pipeline}

In [4]:
error_metrics = ['neg_mean_absolute_error', 'r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_percentage_error']
cv_parameters = KFold(n_splits=10, random_state=1, shuffle=True)

In [5]:
predictor_variables = ['max_precip', 'total_acc',  
                       "hc_at_peak", 'fumax_at_peak', 'lu_at_peak',
                      'se_max', 'se_min',  'sat_at_peak_new', 'sat_at_peak_old',
                      'fu_mean_m',  'fu_min_m', 'fu_max_m', 'fu_at_peak_new', 'fu_at_peak_old', 
                        "mean_sm_2_before_event",
                        'peak_position_ratio','d50','gini','time_based_std','dry_ratio','max_quartile_index',
                      'peak_cell_slope_avg', 'peak_cell_elev_avg','peak_cell_elev_min','peak_cell_slope_max', 
                      #   'peak_cell_elev_max',
                       'peak_cell_water_prop', 'peak_cell_urban_prop', 'peak_cell_inside_prop',
                       "cluster_mean_intensity_t80",
                      "random_variable"]


### Early period

In [6]:
Xfull = subset_early[predictor_variables]
Yfull = subset_early['10cm_area']

# Dataframe to store the scores for all the models
error_metrics_df = pd.DataFrame(columns=['model_name', 'mae', 'mape', 'r2', "rmse"], index=[0])

for i, (model_name, model_pipeline) in enumerate(models_dict.items()):
    print(model_name)
    # Use cross_validate to return the error scores associated with this model and this data
    start = time()
    model_output = cross_validate(model_pipeline, Xfull, Yfull, cv=cv_parameters, scoring=error_metrics, error_score="raise")
    end = time()
    print('Ran in {} minutes'.format(round((end - start)/60),2))
    
    # Formulate the different error scores into a dataframe
    error_metrics_this_df =pd.DataFrame({'model_name': model_name,
                                         'mae': round(abs(model_output['test_neg_mean_absolute_error'].mean()),2), 
                  'mape': round(abs(model_output['test_neg_mean_absolute_percentage_error'].mean()),2),
                  'r2': round(abs(model_output['test_r2'].mean()),2), 
                  'rmse': round(abs(model_output['test_neg_root_mean_squared_error'].mean()),2)}, index=[0])

    # Add evaluation metric scores for this model to the dataframe containing the metrics for each model
    error_metrics_df.loc[i] = error_metrics_this_df.iloc[0]
error_metrics_df    

linear_regressor
Ran in 0 minutes
xgb_regressor
Ran in 0 minutes
rf_regressor
Ran in 2 minutes
et_regressor
Ran in 1 minutes


,model_name,mae,mape,r2,rmse
0,linear_regressor,0.23,40451350994043.640625,0.6,0.35
1,xgb_regressor,0.12,6145275543664.490234,0.88,0.19
2,rf_regressor,0.13,2129281049246.409912,0.85,0.21
3,et_regressor,0.12,2239185087419.040039,0.86,0.2


### Late period

In [7]:
Xfull = subset_late[predictor_variables]
Yfull = subset_late['10cm_area']

# Dataframe to store the scores for all the models
error_metrics_df = pd.DataFrame(columns=['model_name', 'mae', 'mape', 'r2', "rmse"], index=[0])

for i, (model_name, model_pipeline) in enumerate(models_dict.items()):
    print(model_name)
    # Use cross_validate to return the error scores associated with this model and this data
    start = time()
    model_output = cross_validate(model_pipeline, Xfull, Yfull, cv=cv_parameters, scoring=error_metrics, error_score="raise")
    end = time()
    print('Ran in {} minutes'.format(round((end - start)/60),2))
    
    # Formulate the different error scores into a dataframe
    error_metrics_this_df =pd.DataFrame({'model_name': model_name,
                                         'mae': round(abs(model_output['test_neg_mean_absolute_error'].mean()),2), 
                  'mape': round(abs(model_output['test_neg_mean_absolute_percentage_error'].mean()),2),
                  'r2': round(abs(model_output['test_r2'].mean()),2), 
                  'rmse': round(abs(model_output['test_neg_root_mean_squared_error'].mean()),2)}, index=[0])

    # Add evaluation metric scores for this model to the dataframe containing the metrics for each model
    error_metrics_df.loc[i] = error_metrics_this_df.iloc[0]
error_metrics_df    

linear_regressor
Ran in 0 minutes
xgb_regressor
Ran in 0 minutes
rf_regressor
Ran in 3 minutes
et_regressor
Ran in 1 minutes


,model_name,mae,mape,r2,rmse
0,linear_regressor,0.22,52417281997732.0,0.6,0.35
1,xgb_regressor,0.11,8555709098959.929688,0.89,0.19
2,rf_regressor,0.12,2976632256043.22998,0.86,0.2
3,et_regressor,0.12,3259933551092.600098,0.87,0.2


In [8]:
"""Simple hyperparameter tuning for a random forest. Assumes `df` and `predictor_variables` exist."""
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV  # use GroupKFold + groups= if rows aren't independent

TARGET = "10cm_area"
data = df.dropna(subset=[TARGET])
X, y = data[predictor_variables], data[TARGET]

# RF can't handle NaNs: impute or drop them here if your predictors have any
# X = X.fillna(X.median())

param_space = {
    "n_estimators": [300, 500, 800],
    "max_depth": [None, 6, 10, 16, 24],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": [0.2, 0.33, 0.5, 0.75, 1.0],
    "max_samples": [0.5, 0.7, 0.9, None],
}

search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=1),
    param_distributions=param_space,
    n_iter=40,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring="neg_root_mean_squared_error",   # or "r2"
    n_jobs=-1,                               # parallelise across candidates
    random_state=42,
    verbose=1,
)
search.fit(X, y)

print("Best params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

# Top 10 candidates, to see how sensitive the score is to each setting
results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
print(results[["rank_test_score", "mean_test_score", "std_test_score"]
              + [f"param_{k}" for k in param_space]].head(10).to_string())

best_rf = search.best_estimator_   # already refit on all of X, y

Fitting 5 folds for each of 40 candidates, totalling 200 fits


/home/kv25483/anaconda3/envs/geo_env2/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_samples': None, 'max_features': 0.5, 'max_depth': 24}
Best CV RMSE: 0.1807456608084199
    rank_test_score  mean_test_score  std_test_score  param_n_estimators param_max_depth  param_min_samples_leaf  param_max_features param_max_samples
1                 1        -0.180746        0.004147                 300              24                       2                0.50              None
27                2        -0.181491        0.003978                 300              24                       1                1.00               0.9
21                3        -0.183805        0.003640                 300              16                       2                0.75              None
22                4        -0.185088        0.003684                 800              16                       2                1.00              None
9                 5        -0.185232        0.004176                 500            None       

In [9]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_min_samples_leaf,param_max_samples,param_max_features,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
1,1206.917849,14.584692,2.773824,0.380912,300,2,None,0.50,24,"{'n_estimators': 300, 'min_samples_leaf': 2, '...",-0.175527,-0.183722,-0.183485,-0.175899,-0.185094,-0.180746,0.004147,1
27,3289.516232,34.446626,23.554662,11.604705,300,1,0.9,1.00,24,"{'n_estimators': 300, 'min_samples_leaf': 1, '...",-0.176596,-0.186092,-0.183532,-0.176855,-0.184381,-0.181491,0.003978,2
21,1008.175339,45.298865,2.001918,0.080599,300,2,None,0.75,16,"{'n_estimators': 300, 'min_samples_leaf': 2, '...",-0.178808,-0.187225,-0.185960,-0.179998,-0.187035,-0.183805,0.003640,3
22,4993.184343,69.058978,134.905582,19.920882,800,2,None,1.00,16,"{'n_estimators': 800, 'min_samples_leaf': 2, '...",-0.180463,-0.189436,-0.187060,-0.180908,-0.187574,-0.185088,0.003684,4
9,2064.036924,45.652906,4.736473,1.001748,500,2,0.7,1.00,None,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",-0.179968,-0.189700,-0.187351,-0.180439,-0.188704,-0.185232,0.004176,5
11,1740.880501,42.729210,4.129269,0.425264,500,5,None,0.75,24,"{'n_estimators': 500, 'min_samples_leaf': 5, '...",-0.180351,-0.189573,-0.188299,-0.182017,-0.189305,-0.185909,0.003917,6
16,1351.527386,21.609402,2.104418,0.078763,300,5,None,1.00,24,"{'n_estimators': 300, 'min_samples_leaf': 5, '...",-0.181911,-0.192351,-0.189389,-0.182661,-0.189711,-0.187205,0.004152,7
39,513.050963,28.059134,3.634480,0.109839,500,1,0.7,0.33,None,"{'n_estimators': 500, 'min_samples_leaf': 1, '...",-0.182147,-0.191176,-0.189289,-0.181952,-0.193070,-0.187527,0.004630,8
13,2295.617234,44.357415,2999.116313,90.913475,800,1,0.5,0.75,24,"{'n_estimators': 800, 'min_samples_leaf': 1, '...",-0.182769,-0.191635,-0.190746,-0.183358,-0.192403,-0.188182,0.004216,9
23,633.506486,21.096360,5.009253,0.370699,500,2,None,0.20,24,"{'n_estimators': 500, 'min_samples_leaf': 2, '...",-0.183132,-0.194227,-0.190893,-0.183724,-0.195241,-0.189443,0.005121,10
